In [4]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")
# ==============================================================================
# SLIDE 1: IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================================================================
import cv2
import mediapipe as mp

# Inicializa as soluções de desenho e o modelo de detecção de mãos do MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

# ==============================================================================
# SLIDE 2: CONFIGURAÇÃO DO MODELO MEDIAPIPE HANDS
# ==============================================================================
# O MediaPipe Hands processa a imagem identificando as mãos e os seus 21 pontos articulares.
# - static_image_mode=False: Trata o fluxo como vídeo contínuo (otimiza o rastreamento).
# - max_num_hands=1: Limita a detecção a apenas 1 mão para simplificar a análise do sinal.
# - min_detection_confidence=0.5: Confiança mínima para aceitar a detecção inicial da mão.
# - min_tracking_confidence=0.5: Confiança mínima para manter o rastreamento nos quadros seguintes.
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ==============================================================================
# SLIDE 3: CAPTURA DE VÍDEO (WEBCAM)
# ==============================================================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError("Erro ao aceder à webcam.")

print("Iniciando rastreamento de mãos com MediaPipe. Pressione 'q' para sair.")

# ==============================================================================
# SLIDE 4: LAÇO DE PROCESSAMENTO E EXTRAÇÃO DE PONTOS (LANDMARKS)
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        print("Aviso: Não foi possível ler o frame.")
        break

    # O MediaPipe exige que a imagem esteja no formato RGB. O OpenCV lê em BGR.
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Processa o frame para detetar as mãos e extrair as coordenadas dos 21 pontos
    resultado = hands.process(frame_rgb)

    # Se houver mãos detetadas no quadro atual:
    if resultado.multi_hand_landmarks:
        for hand_landmarks in resultado.multi_hand_landmarks:
            
            # Desenha as conexões esqueléticas e os pontos diretamente sobre o frame colorido original
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            # Exemplo de acesso às coordenadas (X, Y) do pulso (Ponto 0) ou ponta do indicador (Ponto 8):
            # h, w, _ = frame.shape
            # x_indicador = int(hand_landmarks.landmark[8].x * w)
            # y_indicador = int(hand_landmarks.landmark[8].y * h)
            # cv2.circle(frame, (x_indicador, y_indicador), 10, (0, 255, 0), cv2.FILLED)

    # Exibe o vídeo processado em tempo real
    cv2.imshow("Reconhecimento de Libras - MediaPipe Hands", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==============================================================================
# SLIDE 5: LIBERAÇÃO DE RECURSOS
# ==============================================================================
cap.release()
cv2.destroyAllWindows()
print("Processamento encerrado.")

Iniciando rastreamento de mãos com MediaPipe. Pressione 'q' para sair.
Processamento encerrado.
